# Heritage Short: ACE-Step 1.5 score candidates (Kaggle)

One instrumental candidate per seed in `music.seeds` (11, 22, 33), from the caption, bpm and key in
`storyboard.json`. Checked against the ACE-Step 1.5 source (`acestep/inference.py`), but **not yet run on a GPU**.

ACE-Step is installed the official way (`uv sync`) into its own environment under `/tmp`, so it does not
fight Kaggle's preinstalled packages and does not fill the 20 GB `/kaggle/working` limit.
On a T4 it runs in float16 with SDPA attention (it detects pre-Ampere GPUs itself).

**Before you start**
1. Use the same `heritage-job` dataset as the Wan notebook (from `make kaggle-export`).
2. Kaggle → *Create → New Notebook* → *File → Import Notebook* → this file.
3. *Add Input* → `heritage-job`. Session options: Accelerator **GPU T4 x2**, Internet **On**.
4. *Run All*. Install and model download take several minutes; each 90 s candidate should then take a few minutes.
5. Download `/kaggle/working/heritage_results_music.zip`, then on your machine:
   `make import ZIP=~/Downloads/heritage_results_music.zip`.
6. Listen to each `outputs/music/candidates/seed_*.wav` under the narration, set `music.chosen_seed` and
   `music.choice_reason` in `storyboard.json`, run `make film`, and log the comparison in EXPERIMENTS.md (E13, E14).

In [ ]:
# 1. GPU check

import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout or "NO GPU: Settings -> Accelerator -> GPU T4 x2")

In [ ]:
# 2. Install ACE-Step 1.5 with uv into /tmp (official method; isolated from the notebook's packages)
%env UV_LINK_MODE=copy
!pip install -q uv
!test -d /tmp/ACE-Step-1.5 || git clone -q --depth 1 https://github.com/ACE-Step/ACE-Step-1.5.git /tmp/ACE-Step-1.5
!cd /tmp/ACE-Step-1.5 && uv sync -q && uv run python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 3. Job, output folder and logger

import glob, json, os, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

found = glob.glob("/kaggle/input/**/jobs.json", recursive=True)
assert found, "jobs.json not found: attach the heritage-job dataset (Add input) and re-run this cell"
JOB = Path(found[0]).parent
jobs = json.loads((JOB / "jobs.json").read_text())
OUT = Path("/kaggle/working/results")
OUT.mkdir(parents=True, exist_ok=True)
LOG = OUT / "runs_kaggle.jsonl"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-kaggle"

def gpu_name():
    try:
        import torch
        return torch.cuda.get_device_name(0)
    except Exception:
        return "unknown"

def log_call(stage, shot, model, params, seed, latency_s, status, output=None, error=None):
    """Same shape as pipeline/common.py log lines, so `make import` can merge them."""
    rec = {"ts": datetime.now(timezone.utc).isoformat(timespec="milliseconds"), "run_id": RUN_ID,
           "event": "model_call", "stage": stage, "shot": shot, "provider": "kaggle", "model": model,
           "params": params, "seed": seed, "output": output, "attempt": 1, "status": status,
           "latency_s": round(latency_s, 2), "gpu": gpu_name(),
           "cost": {"gpu_minutes": round(latency_s / 60, 2), "tier": "free (Kaggle weekly GPU quota)"}}
    if error:
        rec["error"] = str(error)[:300]
    with LOG.open("a") as f:
        f.write(json.dumps(rec) + "\n")
    print(f"{status:5} {shot or ''} {latency_s:7.1f}s {output or error or ''}")

print("job folder:", JOB, "| storyboard version", jobs["storyboard_version"])

In [ ]:
# 4. The generation script (runs inside ACE-Step's environment)
GEN = r'''
# Runs inside ACE-Step's own uv environment. Writes one WAV per seed and one log line per attempt.
import json, shutil, sys, time
from datetime import datetime, timezone
from pathlib import Path

REPO, JOB, OUT, RUN_ID = Path(sys.argv[1]), Path(sys.argv[2]), Path(sys.argv[3]), sys.argv[4]
sys.path.insert(0, str(REPO))
import torch
from acestep.handler import AceStepHandler
from acestep.inference import GenerationConfig, GenerationParams, generate_music

M = json.loads((JOB / "jobs.json").read_text())["music"]
DIT = "acestep-v15-turbo"
LOG = OUT / "runs_kaggle.jsonl"
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"

def log_call(params, seed, latency_s, status, output=None, error=None):
    rec = {"ts": datetime.now(timezone.utc).isoformat(timespec="milliseconds"), "run_id": RUN_ID,
           "event": "model_call", "stage": "music", "shot": None, "provider": "kaggle",
           "model": "ACE-Step 1.5 " + DIT, "params": params, "seed": seed, "output": output, "attempt": 1,
           "status": status, "latency_s": round(latency_s, 2), "gpu": GPU,
           "cost": {"gpu_minutes": round(latency_s / 60, 2), "tier": "free (Kaggle weekly GPU quota)"}}
    if error:
        rec["error"] = str(error)[:300]
    with LOG.open("a") as f:
        f.write(json.dumps(rec) + "\n")
    print(f"{status:5} seed {seed} {latency_s:7.1f}s {output or error or ''}", flush=True)

dit = AceStepHandler()
status, ok = dit.initialize_service(project_root=str(REPO), config_path=DIT, device="cuda")
print(status, flush=True)
if not ok:
    sys.exit("ACE-Step failed to initialise; see the message above")

(OUT / "music").mkdir(parents=True, exist_ok=True)
for seed in M["seeds"]:
    dst = OUT / "music" / f"seed_{seed}.wav"
    if dst.exists():
        print("skip (done) seed", seed); continue
    params = {"caption": M["caption"], "bpm": M["bpm"], "keyscale": M["keyscale"], "duration": M["duration"],
              "instrumental": True, "inference_steps": 8, "thinking": False, "dit": DIT}
    t0 = time.perf_counter()
    try:
        result = generate_music(
            dit, None,   # no LM planner: caption, bpm and key are given explicitly
            GenerationParams(caption=M["caption"], lyrics="[Instrumental]", instrumental=True,
                             duration=M["duration"], bpm=M["bpm"], keyscale=M["keyscale"],
                             seed=seed, inference_steps=8, thinking=False),
            GenerationConfig(batch_size=1, audio_format="wav", use_random_seed=False, seeds=[seed]),
            save_dir=str(OUT / "tmp"))
        if not result.success:
            raise RuntimeError(result.error)
        shutil.copy(result.audios[0]["path"], dst)
        log_call(params, seed, time.perf_counter() - t0, "ok", output=f"outputs/music/candidates/{dst.name}")
    except Exception as err:
        log_call(params, seed, time.perf_counter() - t0, "error", error=err)
shutil.rmtree(OUT / "tmp", ignore_errors=True)
'''
Path('/tmp/gen_music.py').write_text(GEN)
print('written /tmp/gen_music.py')

In [ ]:
# 5. Generate the three candidates (checkpoints download automatically on first run)
!cd /tmp/ACE-Step-1.5 && uv run python /tmp/gen_music.py /tmp/ACE-Step-1.5 "{JOB}" "{OUT}" "{RUN_ID}"

In [ ]:
# 6. Listen here
from IPython.display import Audio, display
for f in sorted((OUT / "music").glob("seed_*.wav")):
    print(f.name); display(Audio(str(f)))

In [ ]:
# 7. Zip results for download
ZIP_NAME = "heritage_results_music.zip"

# One download brings everything home. On your machine: make import ZIP=~/Downloads/<this file>
zpath = Path("/kaggle/working") / ZIP_NAME
with zipfile.ZipFile(zpath, "w") as z:
    for f in OUT.rglob("*"):
        if f.is_file():
            z.write(f, f.relative_to(OUT.parent))
print("download:", zpath, round(zpath.stat().st_size / 1e6, 1), "MB")
print(open(LOG).read() if LOG.exists() else "no log lines")